In [1]:
# Simple Repr 
'''
START
  ↓
Chatbot
  ↓
Need Tool?
 ├── No → END
 │
 └── Yes
       ↓
    ToolNode
       ↓
    Chatbot'''

'\nSTART\n  ↓\nChatbot\n  ↓\nNeed Tool?\n ├── No → END\n │\n └── Yes\n       ↓\n    ToolNode\n       ↓\n    Chatbot'

In [2]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.messages import HumanMessage

In [3]:
### Tools

@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two numbers.
    """
    return a * b


@tool
def add(a: int, b: int) -> int:
    """
    Add two numbers.
    """
    return a + b


@tool
def square(number: int) -> int:
    """
    Return square of a number.
    """
    return number * number

In [4]:
tools = [multiply,add, square]

In [5]:
llm = ChatGroq(model = 'llama-3.3-70b-versatile')

In [6]:
llm_with_tools = llm.bind_tools(tools)

In [7]:
def chatbot(state: MessagesState):

    response = llm_with_tools.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }

In [8]:
# ToolNode executes tool calls generated by the LLM.

tool_node = ToolNode(tools)

In [9]:
builder = StateGraph(MessagesState)

builder.add_node("chatbot",chatbot)

builder.add_node("tools",tool_node)

In [ ]:
"""
Chatbot
 ↓
Need Tool?
"""

"""
tools_condition decides:
    Tool Needed?
        YES -> tools
        NO -> END
"""

"""
After tool execution:
    Tool
    ↓
    Chatbot
    This creates the Agent Loop.
"""

In [10]:
builder.add_edge(START,"chatbot")

builder.add_conditional_edges("chatbot",tools_condition)

builder.add_edge("tools","chatbot")

In [11]:
graph = builder.compile()

In [12]:
# no tool call here

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="What is LangGraph?"
            )
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)
    print("-" * 50)

HumanMessage
What is LangGraph?
--------------------------------------------------
AIMessage
LangGraph is an AI model developed by Meta, designed to process and generate human-like language. It is a type of large language model (LLM) that uses natural language processing (NLP) to understand and respond to user input. LangGraph is trained on a massive dataset of text from various sources, including books, articles, and conversations, which enables it to learn patterns and relationships in language. This training allows LangGraph to generate coherent and context-specific text, making it useful for applications such as chatbots, language translation, and text summarization.
--------------------------------------------------


In [13]:
## Tool call example

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="What is 25 multiplied by 12?"
            )
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)

    if hasattr(message, "content"):
        print(message.content)

    print("-" * 50)

HumanMessage
What is 25 multiplied by 12?
--------------------------------------------------
AIMessage

--------------------------------------------------
ToolMessage
300
--------------------------------------------------
AIMessage
The result of 25 multiplied by 12 is 300.
--------------------------------------------------


In [14]:
## Multiple tool calls

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
                Add 10 and 20.
                Then square the result.
                """
            )
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)

    if hasattr(message, "content"):
        print(message.content)

    print("-" * 50)

HumanMessage

                Add 10 and 20.
                Then square the result.
                
--------------------------------------------------
AIMessage

--------------------------------------------------
ToolMessage
30
--------------------------------------------------
ToolMessage
900
--------------------------------------------------
AIMessage
The sum of 10 and 20 is 30. The square of 30 is 900.
--------------------------------------------------
